In [41]:
import requests
import os
import sys
import platform
from lakehouse.spark import bronze, silver
from pyspark.sql import DataFrame, SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
import json

In [42]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [43]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [44]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up and Bronze Data

In [45]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

In [46]:
options = {"catalog": CATALOG, "target_schema": "bronze"}

In [47]:
@F.udf(returnType="STRING")
def get_properties(url):
    json_request = requests.get(url).json()
    return json.dumps(json_request["result"]["properties"])

In [48]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("properties", get_properties(F.col("url")))


bronze_instance = StarWarsBronze(spark, **options)

In [49]:
bronze_instance.load().transform().write(mode="overwrite").execute("people", "planets")

2025-03-16 07:25:01 | people | execute | Started
2025-03-16 07:25:01 | people | execute | Started
2025-03-16 07:25:01 | people | load | Started
2025-03-16 07:25:05 | people | load | Completed in 0.07 min
2025-03-16 07:25:05 | people | transform | Started
2025-03-16 07:25:05 | people | transform | Completed in 0.0 min
2025-03-16 07:25:05 | people | write | Started
2025-03-16 07:25:22 | people | write | Completed in 0.28 min
2025-03-16 07:25:22 | people | execute | Completed in 0.35 min
2025-03-16 07:25:22 | planets | execute | Started
2025-03-16 07:25:22 | planets | load | Started
2025-03-16 07:25:25 | planets | load | Completed in 0.03 min
2025-03-16 07:25:25 | planets | transform | Started
2025-03-16 07:25:25 | planets | transform | Completed in 0.0 min
2025-03-16 07:25:25 | planets | write | Started
2025-03-16 07:25:39 | planets | write | Completed in 0.22 min
2025-03-16 07:25:39 | planets | execute | Completed in 0.27 min
2025-03-16 07:25:39 | people | execute | Completed in 0.63 mi

In [50]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show()

No. Rows: 82
+--------------------+--------------------+---+--------------------+--------------------+
|         LH_BronzeTS|                name|uid|                 url|          properties|
+--------------------+--------------------+---+--------------------+--------------------+
|2025-03-16 07:25:...|           Boba Fett| 22|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|               IG-88| 23|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|               Bossk| 24|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|    Lando Calrissian| 25|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|               Lobot| 26|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|              Ackbar| 27|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|          Mon Mothma| 28|https://www.swapi...|{"created": "2025...|
|2025-03-16 07:25:...|        Arvel Crynyd| 29|https://www.swapi...|{"created": "2025..

In [51]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(truncate=False)

No. Rows: 60
+--------------------------+--------------+---+-------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|LH_BronzeTS               |name          |uid|url                                  |properties                                                                                                                                                                                                                                                                                                                                                                                    |
+--------------------------+--------------+---+--

In [52]:
ddl_schema_planets = "diameter STRING, rotation_period STRING, orbital_period STRING, gravity STRING, population STRING, climate STRING, terrain STRING, surface_water STRING, created STRING, edited STRING, name STRING, url STRING"
ddl_schema_people = "height STRING, mass STRING, hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING, created STRING, edited STRING, name STRING, homeworld STRING, url STRING"
DDL_SCHEMAS = {"planets": ddl_schema_planets, "people": ddl_schema_people}

In [53]:
options = {
    "catalog": CATALOG,
    "source_schema": "bronze",
    "target_schema": "silver",
}

# 2 Overwrite

In [54]:
config = {
    "load": {
        "mode": "default",
        "filter": "all",
        "date_col": "LH_BronzeTS",
    },
    "transform": {
        "ignore_defaults": False,
        "transformation_order": [
            "add_dummy_col",
            "rename_columns",
            "tbl_transformations",
            "select_columns",
            "cast_column_types",
        ],
        # "tbl_transformations": {"tbl": "custom_transform1"},
        "rename_columns": {
            "planets": {"dummy_col": "dummy"},
            "people": {"dummy_col": "dummy"},
        },
        "select_columns": {
            "planets": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
            "people": ["LH_BronzeTS", "name", "uid", "url", "dummy"],
        },
        "cast_column_types": {
            "planets": {"dummy": "string", "id": "int"},
            "people": {"dummy": "string", "id": "int"},
        },
    },
    "write": {
        "mode": "overwrite",
        "merge_schema": True,
        "external": False,
    },
    "optimize": {
        "optimize": True,
        "optimize_full": False,
        "vacuum": True,
        "vacuum_lite": True,
        "analyze": False,
        "retention": 168,
        # "excl_cols": ["a", "b"],
    },
    "tblproperties": {
        # "clusterby": ["a", "b"],
        "deletion_vectors": True,
        "auto_compact": True,
        "optimize_write": True,
        "change_data_feed": True,
        "row_tracking": True,
        "type_widening": False,
        "tblproperties": {"enableChangeDataFeed": "true"},
    },
}

In [55]:
class StarWarsSilver(silver.Silver):
    def custom_filter(self, df: DataFrame, table: str) -> DataFrame:
        return df.where("uid <= '25'")

    def custom_transform(self, df: DataFrame, table: str) -> DataFrame:
        df = df.withColumn(
            "properties", F.from_json(F.col("properties"), DDL_SCHEMAS[table])
        )
        df = df.withColumn("uid", F.col("uid").cast("int"))
        return df

    def add_dummy_col(self, df: DataFrame, table: str) -> DataFrame:
        return df.withColumn("dummy_col", F.lit("dummy"))


silver_instance = StarWarsSilver(
    spark,
    catalog=CATALOG,
    source_schema="bronze",
    target_schema="silver",
    config=config,
)

In [56]:
silver_instance.execute("people", "planets")

2025-03-16 07:25:41 | people | execute | Started
2025-03-16 07:25:41 | people | execute | Started
2025-03-16 07:25:41 | people | load | Started
2025-03-16 07:25:41 | people | load | Completed in 0.0 min
2025-03-16 07:25:41 | people | transform | Started
2025-03-16 07:25:41 | people | transform | Completed in 0.0 min
2025-03-16 07:25:41 | people | write | Started
2025-03-16 07:25:43 | people | write | Completed in 0.02 min
2025-03-16 07:25:43 | people | tblproperties | Started
2025-03-16 07:25:56 | people | tblproperties | Completed in 0.22 min
2025-03-16 07:25:56 | people | optimize | Started
2025-03-16 07:26:07 | people | optimize | Completed in 0.18 min
2025-03-16 07:26:07 | people | execute | Completed in 0.43 min
2025-03-16 07:26:07 | planets | execute | Started
2025-03-16 07:26:07 | planets | load | Started
2025-03-16 07:26:07 | planets | load | Completed in 0.0 min
2025-03-16 07:26:07 | planets | transform | Started
2025-03-16 07:26:07 | planets | transform | Completed in 0.0 min

In [57]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.people")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 82
+--------------------------+--------------------------+---------------------+---+------------------------------------+-----+
|LH_SilverTS               |LH_BronzeTS               |name                 |uid|url                                 |dummy|
+--------------------------+--------------------------+---------------------+---+------------------------------------+-----+
|2025-03-16 07:25:41.886253|2025-03-16 07:25:05.801266|Cliegg Lars          |62 |https://www.swapi.tech/api/people/62|dummy|
|2025-03-16 07:25:41.886253|2025-03-16 07:25:05.801266|Poggle the Lesser    |63 |https://www.swapi.tech/api/people/63|dummy|
|2025-03-16 07:25:41.886253|2025-03-16 07:25:05.801266|Luminara Unduli      |64 |https://www.swapi.tech/api/people/64|dummy|
|2025-03-16 07:25:41.886253|2025-03-16 07:25:05.801266|Barriss Offee        |65 |https://www.swapi.tech/api/people/65|dummy|
|2025-03-16 07:25:41.886253|2025-03-16 07:25:05.801266|Dormé                |66 |https://www.swapi.tech/api/peop

In [58]:
df = spark.sql(f"SELECT * FROM {CATALOG}.silver.planets")
print(f"No. Rows: {df.count()}")
df.show(100, truncate=False)

No. Rows: 60
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----+
|LH_SilverTS               |LH_BronzeTS               |name          |uid|url                                  |dummy|
+--------------------------+--------------------------+--------------+---+-------------------------------------+-----+
|2025-03-16 07:26:08.005438|2025-03-16 07:25:25.924776|Mygeeto       |16 |https://www.swapi.tech/api/planets/16|dummy|
|2025-03-16 07:26:08.005438|2025-03-16 07:25:25.924776|Felucia       |17 |https://www.swapi.tech/api/planets/17|dummy|
|2025-03-16 07:26:08.005438|2025-03-16 07:25:25.924776|Cato Neimoidia|18 |https://www.swapi.tech/api/planets/18|dummy|
|2025-03-16 07:26:08.005438|2025-03-16 07:25:25.924776|Saleucami     |19 |https://www.swapi.tech/api/planets/19|dummy|
|2025-03-16 07:26:08.005438|2025-03-16 07:25:25.924776|Stewjon       |20 |https://www.swapi.tech/api/planets/20|dummy|
|2025-03-16 07:26:08.005438|2025-03

In [59]:
df = spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.planets")
df.show(100, truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                                                                                                                                             |job |notebook|clusterId|readVersion|isolationLevel   |isBlindAppend|op

# 6 Clean Up

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.silver CASCADE")
spark.stop()

DataFrame[]